# 06. 80 Expert 모델: 수학 1문제와 LCB 2문제

기존에 비교한 MATH-500 11번 다항식 문제(`math500-test-11`, 정답 `3/56`)와 LCB `3674`, `3677`을 시험합니다. 세 문제는 모델을 한 번만 적재해 생성하고, 모델 종료 후 채점합니다. 이는 전체 벤치마크 점수가 아닙니다.

이전 수학 실행은 context 16,384 / 최대 생성 8,192였고, 이 실행은 context 요청 65,535 / 최대 생성 32,768입니다. 문제는 같지만 속도와 점수를 동일 조건으로 비교할 수 없습니다. 세 문제라도 시간이 오래 걸릴 수 있습니다.

## Colab 환경 자동 준비
바로 아래 설정에서 `SETUP_COLAB = True`로 변경하고 위에서부터 실행하세요.
모델은 별도로 준비합니다. 기본 경로: `/content/Putare/models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf`
`MODEL_PATH`에 다른 모델 경로를 지정할 수 있습니다. 최초 실행은 의존성 설치와 CUDA 엔진 빌드를 수행합니다.
소스와 채점 자료를 내장해 노트북 크기가 약 77MB이며 Colab 웹 편집기의 업로드·실행은 검증되지 않았습니다.


In [2]:
# 2026-11-09: 설치와 빌드는 명시적으로 켰을 때만 실행한다.
SETUP_COLAB = True
MODEL_PATH = None  # None이면 /content/Putare/models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf
if SETUP_COLAB:
    _PUTARE_SETUP_READY = False


In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

upload = FileUpload(accept='.gguf')
display(upload)

In [ ]:
def restore_embedded_bundle(root, encoded, expected_sha256):
    # 2026-11-09: 검증과 충돌 확인을 모두 마친 후에만 사용자 파일을 생성한다.
    import base64, hashlib, io, json, zipfile
    from pathlib import Path, PurePosixPath
    root = Path(root)
    if root.is_symlink():
        raise ValueError('복원 경로가 심볼릭 링크입니다')
    try:
        raw = base64.b64decode(encoded, validate=True)
        if hashlib.sha256(raw).hexdigest() != expected_sha256:
            raise ValueError('내장 묶음 SHA256 불일치')
        archive = zipfile.ZipFile(io.BytesIO(raw))
        names = archive.namelist()
        if len(names) != len(set(names)) or '_putare_inventory.json' not in names:
            raise ValueError('내장 묶음 파일 목록 오류')
        expected = json.loads(archive.read('_putare_inventory.json'))
        if not isinstance(expected, dict) or set(names) != set(expected) | {'_putare_inventory.json'}:
            raise ValueError('내장 묶음 인벤토리 불일치')
        for name, digest in expected.items():
            p = PurePosixPath(name)
            if p.is_absolute() or not p.parts or any(x in ('', '.', '..') for x in name.split('/')) or '\\' in name or ':' in name:
                raise ValueError('안전하지 않은 묶음 경로')
            info = archive.getinfo(name)
            if info.is_dir() or (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('묶음에 링크 또는 디렉터리가 있습니다')
            if hashlib.sha256(archive.read(name)).hexdigest() != digest:
                raise ValueError('내장 파일 SHA256 불일치: ' + name)
            target = root.joinpath(*p.parts)
            for part in [root, *[root.joinpath(*p.parts[:i]) for i in range(1, len(p.parts) + 1)]]:
                if part.is_symlink():
                    raise ValueError('복원 경로에 심볼릭 링크가 있습니다: ' + str(part))
                if part != target and part.exists() and not part.is_dir():
                    raise FileExistsError('복원 폴더 경로에 파일이 있습니다: ' + str(part))
            if target.exists() and (not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != digest):
                raise FileExistsError('기존 파일이 달라 덮어쓰지 않습니다: ' + str(target))
        for name in expected:
            target = root.joinpath(*PurePosixPath(name).parts)
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open('xb') as output:
                    output.write(archive.read(name))
    except (zipfile.BadZipFile, KeyError, UnicodeError) as error:
        raise ValueError('내장 묶음이 손상되었습니다') from error
    return len(expected)


In [ ]:
if SETUP_COLAB:
    _BUNDLE_DATA = (
        '...'
)
    _BUNDLE_SHA256 = '4427c866577673f529cdb4487264d5bdf15e2f58071a964648ad4341e35efa4e'


In [ ]:
if SETUP_COLAB:
    _PUTARE_SETUP_READY = False
    import os, shutil, subprocess, sys
    from pathlib import Path
    ROOT = Path('/content/Putare')
    if not (3, 12) <= sys.version_info[:2] < (3, 13):
        raise RuntimeError('Python 3.12가 필요합니다. Colab 런타임 버전을 확인하세요.')
    missing = [name for name in ('git', 'cmake', 'ninja', 'nvcc', 'c++') if not shutil.which(name)]
    if missing:
        raise RuntimeError('필요한 빌드 도구가 없습니다: ' + ', '.join(missing) + '. Colab에 git, cmake, ninja, C++ 컴파일러와 CUDA Toolkit을 설치하세요.')
    restored = restore_embedded_bundle(ROOT, _BUNDLE_DATA, _BUNDLE_SHA256)
    print(f'소스와 자료 {restored}개 확인 완료')
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    subprocess.run([sys.executable, '-m', 'pip', '--isolated', 'install', '-r', str(ROOT / 'requirements-research.txt')], check=True)
    from tools.native.gpu_memory import NvmlDevice
    device = NvmlDevice()
    try:
        major, minor = device.nvml.nvmlDeviceGetCudaComputeCapability(device.handle)
        target = f'{major}{minor}'
        print(f'GPU: {device.name}, Compute Capability: {major}.{minor}')
    finally:
        device.close()
    codes = subprocess.run(['nvcc', '--list-gpu-code'], check=True, capture_output=True, text=True).stdout.split()
    if f'sm_{target}' not in codes:
        raise RuntimeError(f'현재 nvcc가 sm_{target}을 지원하지 않습니다. GPU를 지원하는 CUDA Toolkit이 필요합니다.')
    # 기존 소스/빌드가 있으면 prepare가 고정 revision과 오염 여부를 검증한다.
    subprocess.run([sys.executable, str(ROOT / 'tools/native/prepare.py'), '--backend', 'cuda', '--gpu-target', target, '--execute'], cwd=ROOT, check=True)
    from moe_prune.benchmark_data import prepare_grader
    prepare_grader(ROOT)
    _PUTARE_SETUP_READY = True
    _expected_model = Path(MODEL_PATH) if MODEL_PATH else ROOT / 'models/gemma-4-26B-A4B-it-UD-IQ4_NL-lcb100-activity-ordered-e80.gguf'
    print('환경 준비 완료. 모델 경로:', _expected_model, '| 존재:', _expected_model.is_file())
    if not _expected_model.is_file():
        print('추론 전 모델 파일을 위 경로에 준비하세요.')


In [ ]:
from pathlib import Path
import sys
import subprocess

ROOT = Path.cwd().resolve()
if ROOT.name == 'ipynb': ROOT = ROOT.parent
assert (ROOT / 'moe_prune').is_dir(), '저장소 루트 또는 ipynb 폴더에서 실행하세요.'
sys.path.insert(0, str(ROOT))
from tools import benchmark80 as trial
from tools import manual80

INSTALL_DEPENDENCIES = False
PREPARE_GRADER = False
PREPARE_ENGINE = False
BUILD_ENGINE = False
RUN_GENERATION = False
RUN_GRADING = False
RUN_SHOW_TEXT = False
MODEL = Path(MODEL_PATH) if MODEL_PATH else manual80.MODEL
ENGINE = manual80.ENGINE
RUN_DIR = trial.DIRECTORY
GRADER_VENV = ROOT / '.cache/putare-lcb/venv'

print({'model': str(MODEL), 'model_present': MODEL.is_file(),
       'engine': str(ENGINE), 'engine_present': ENGINE.is_file(),
       'run_dir': str(RUN_DIR), 'run_exists': RUN_DIR.exists(), 'case_ids': trial.IDS})

## 환경 준비

Colab A100 커널과 저장소·80 Expert GGUF·05번 manifest·기존 LCB 묶음이 필요합니다. 필요한 플래그를 켜 Python 의존성, CUDA 엔진, 공식 채점기를 준비합니다. 기존 PyTorch는 교체하지 않습니다.

In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', '--isolated', 'install', '-r', str(ROOT / 'requirements-research.txt')], check=True)
if PREPARE_ENGINE:
    command = [sys.executable, str(ROOT / 'tools/native/prepare.py'), '--backend', 'cuda']
    if BUILD_ENGINE: command.append('--execute')
    subprocess.run(command, cwd=ROOT, check=True)
if PREPARE_GRADER:
    from moe_prune.benchmark_data import prepare_grader
    prepare_grader(ROOT)
if not any((INSTALL_DEPENDENCIES, PREPARE_ENGINE, PREPARE_GRADER)):
    print('환경 준비를 실행하지 않았습니다.')

## 답안 생성

`RUN_GENERATION=True`로 설정합니다. 기존 `RUN_DIR`은 덮어쓰지 않으므로 다시 실행하려면 사용하지 않은 경로를 지정하세요. 전체 GPU 적재를 요청하며 RAM 80%, VRAM 97% 초과 시 경고 후 계속합니다. 모델·데이터 출처, 실제 실행 설정과 80 Expert 라우팅을 검증합니다.

In [ ]:
if RUN_GENERATION:
    if not globals().get('_PUTARE_SETUP_READY', True):
        raise RuntimeError('환경 준비가 완료되지 않았습니다. 준비 셀을 다시 실행하세요.')
    if not MODEL.is_file():
        raise FileNotFoundError(f'모델 파일을 준비하세요: {MODEL}')
    # 2026-11-09: 모델 산출물에 실제 사용한 GPU 정보를 함께 기록한다.
    from tools.native.gpu_memory import NvmlDevice
    device = NvmlDevice()
    try:
        major, minor = device.nvml.nvmlDeviceGetCudaComputeCapability(device.handle)
        gpu_info = {'name': device.name, 'uuid': device.uuid, 'compute_capability': f'{major}.{minor}'}
    finally:
        device.close()
    import json
    result = trial.generate(RUN_DIR, model=MODEL, engine=ENGINE)
    with (RUN_DIR / 'gpu-info.json').open('x', encoding='utf-8') as output:
        json.dump(gpu_info, output, ensure_ascii=False, indent=2)
    print(result)
else:
    print('생성 실행 대기 중입니다.')


## 채점

`RUN_GRADING=True`는 세 문제 생성과 모델 종료 후 실행합니다. LCB는 고정된 공식 채점기를 사용합니다. 생성된 코드는 별도 프로세스에서 Colab 사용자 권한으로 실행되며 보안 격리되지 않습니다. 수학은 최종 `\boxed{...}`의 분수를 정확히 비교합니다. 인식하지 못한 답은 수동 확인으로 남기며 공식 MATH-500 점수가 아닙니다.

In [ ]:
if RUN_GRADING:
    if not globals().get('_PUTARE_SETUP_READY', True):
        raise RuntimeError('환경 준비가 완료되지 않았습니다.')
    print(trial.grade(RUN_DIR, model=MODEL, engine=ENGINE, venv=GRADER_VENV))
else:
    print('채점 대기 중입니다.')

## 결과 확인

생성 토큰 수, 실행 시간, decode 속도와 표본 메모리를 표시합니다. 수학 결과가 `unparsed`이면 원문을 직접 확인하세요. 생성 원문은 `RUN_SHOW_TEXT=True`일 때 표시합니다.

In [ ]:
import json
from IPython.display import Markdown, display
status_path = RUN_DIR / 'status.json'
summary_path = RUN_DIR / 'grading/summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.is_file() else {}
by_id = {row['id']: row for row in summary.get('cases', [])}
if status_path.is_file():
    table = ['| 문제 | 결과 | 생성 토큰 | 시간(초) | decode tok/s | RAM GiB | GPU 표본 GiB | 수학 답 |',
             '|---|---|---:|---:|---:|---:|---:|---|']
    def fmt(value, scale=1):
        return '미측정' if value is None else f'{value / scale:.2f}'
    for number, ident in enumerate(trial.IDS, 1):
        path = RUN_DIR / 'answers' / f'{number:03d}.json'
        if not path.is_file():
            table.append(f'| {ident} | 생성 대기 | - | - | - | - | - | - |')
            continue
        answer = json.loads(path.read_text(encoding='utf-8'))
        row = by_id.get(ident, {})
        state = '통과' if row.get('passed') is True else '실패' if row.get('passed') is False else row.get('status', '채점 대기')
        measurement, timing = answer.get('measurement') or {}, answer.get('timings') or {}
        gpu = measurement.get('os_gpu_memory') or {}
        table.append(f"| {ident} | {state} | {len(answer.get('token_ids', []))} | {fmt(measurement.get('wall_s'))} | {fmt(timing.get('decode_tokens_s'))} | {fmt(measurement.get('peak_process_tree_rss_bytes'), 1024**3)} | {fmt(gpu.get('peak_bytes'), 1024**3)} | {row.get('answer') or '-'} |")
        if RUN_SHOW_TEXT:
            print(f'\n[{ident}]\n{answer.get("text", "")}')
    display(Markdown('\n'.join(table)))
else:
    print('아직 이 실행의 생성 결과가 없습니다.')